# Content-Based Recommender — TF-IDF on Genres and Tags

Builds a content-based recommender using item metadata (genres + user tags). Not affected by rating frequency imbalances, so it works as a sparsity-robust baseline in `05_sparsity_experiment.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

DATA_DIR    = '../../ml-25m/'
OUTPUT_DIR  = '../outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# load metadata
# movies.csv  →  movieId | title | genres (pipe-separated)
# tags.csv    →  userId  | movieId | tag | timestamp
print('Loading movies.csv and tags.csv...')
movies = pd.read_csv(DATA_DIR + 'movies.csv')
tags   = pd.read_csv(DATA_DIR + 'tags.csv')

print(f'  movies : {movies.shape}')
print(f'  tags   : {tags.shape}')

In [ ]:
# combine genres + tags into one text doc per film
# aggregate tags per movie
movie_tags = (
    tags.groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x.astype(str).str.lower().str.strip()))
    .reset_index()
    .rename(columns={'tag': 'tags_text'})
)

movies_feat = (
    movies
    .merge(movie_tags, on='movieId', how='left')
    .assign(tags_text   = lambda d: d['tags_text'].fillna(''))
    .assign(genre_text  = lambda d: d['genres'].str.replace('|', ' ', regex=False)
                                               .str.replace('(no genres listed)', '', regex=False))
    .assign(content     = lambda d: (d['genre_text'] + ' ' + d['tags_text']).str.strip())
)

has_tags = (movies_feat['tags_text'] != '').sum()
print(f'Films with ≥1 tag : {has_tags:,} / {len(movies_feat):,}')
print('\nSample content doc (Forrest Gump):')
print(movies_feat.loc[movies_feat['title'].str.startswith('Forrest'), 'content'].values[0][:200])

In [ ]:
print('Fitting TF-IDF vectoriser...')
tfidf_vec = TfidfVectorizer(
    max_features  = 10_000,
    min_df        = 2,
    stop_words    = 'english',
    ngram_range   = (1, 2),
    sublinear_tf  = True,
)
raw_matrix   = tfidf_vec.fit_transform(movies_feat['content'])
tfidf_matrix = normalize(raw_matrix, norm='l2')

movie_ids       = movies_feat['movieId'].values
movie_id_to_idx = {int(mid): i for i, mid in enumerate(movie_ids)}

print(f'TF-IDF matrix : {tfidf_matrix.shape}  (movies × features)')
print(f'Vocabulary    : {len(tfidf_vec.vocabulary_):,} terms')
print(f'Matrix nnz    : {tfidf_matrix.nnz:,}  (sparsity '
      f'{1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4%})')

In [ ]:
def get_content_based_recommendations(
    user_id, train_df, tfidf_mat, mids, mid_to_idx, n=10
):
    """
    Build a weighted TF-IDF profile from seed ratings, return top-N by cosine sim.
    Each seed film is weighted by rating/5.0 so highly-rated items drive the profile more.
    Returns empty list if user has no training ratings or produces an all-zero profile.
    """
    user_df = train_df[train_df['userId'] == user_id]
    if user_df.empty:
        return []

    seen_ids = set(user_df['movieId'].astype(int))

    profile = np.zeros(tfidf_mat.shape[1], dtype=np.float32)
    for _, row in user_df.iterrows():
        mid = int(row['movieId'])
        if mid in mid_to_idx:
            weight   = float(row['rating']) / 5.0
            profile += weight * tfidf_mat[mid_to_idx[mid]].toarray().ravel()

    norm = np.linalg.norm(profile)
    if norm == 0:
        return []
    profile /= norm

    sims = tfidf_mat.dot(profile)

    ranked = sorted(
        [(int(mids[i]), float(sims[i]))
         for i in range(len(mids)) if int(mids[i]) not in seen_ids],
        key=lambda x: x[1],
        reverse=True,
    )
    return [mid for mid, _ in ranked[:n]]


print('get_content_based_recommendations() defined.')

In [ ]:
# quick sanity check on a real user

print('Running smoke test...')
ratings_sample = pd.read_csv(DATA_DIR + 'ratings.csv', nrows=200_000)
test_user_id   = int(ratings_sample['userId'].value_counts().index[0])
user_train_df  = ratings_sample[ratings_sample['userId'] == test_user_id]

recs = get_content_based_recommendations(
    test_user_id, user_train_df, tfidf_matrix, movie_ids, movie_id_to_idx, n=10
)

title_map = dict(zip(movies['movieId'], movies['title']))
print(f'\nUser {test_user_id} — {len(user_train_df)} seed ratings')
print('\nSeed films (top-rated):')
for _, r in user_train_df.nlargest(5, 'rating').iterrows():
    print(f'  {r["rating"]}★  {title_map.get(int(r["movieId"]), "Unknown")}')
print('\nContent-based recommendations:')
for i, mid in enumerate(recs, 1):
    print(f'  {i:2d}. {title_map.get(mid, "Unknown")}')

In [ ]:
# save everything needed by 05_sparsity_experiment to score without re-fitting.

cb_model_artefact = {
    'tfidf_matrix'    : tfidf_matrix,
    'tfidf_vectorizer': tfidf_vec,
    'movie_ids'       : movie_ids,
    'movie_id_to_idx' : movie_id_to_idx,
}

out_path = OUTPUT_DIR + 'content_based_model.pkl'
joblib.dump(cb_model_artefact, out_path, compress=3)
print(f'Content-based model saved  →  {out_path}')
print(f'File size: {os.path.getsize(out_path) / 1e6:.1f} MB')